In [1]:
import pickle,re,yaml,os
import pandas as pd
import numpy as np

def ding():
    os.system('afplay /System/Library/Sounds/Submarine.aiff')

In [2]:
repFiles="/Users/gilles/sDrive/Recherche/Boye/HDR/Data/vlexique2.0.3/WordStructure/"

#### paireClasses

In [3]:
class paireClasses:
    def __init__(self,case1,case2):
        self.case1=case1
        self.case2=case2
        self.nom=case1+"-"+case2
        self.classes1=classesPaire(case1,case2)
        self.classes2=classesPaire(case2,case1)

    def ajouterPatron(self,n,patron,motif):
        if n==1:
            self.classes1.ajouterPatron(patron,motif)
        elif n==2:
            self.classes2.ajouterPatron(patron,motif)
        else:
            if debug: print ("le numéro de forme n'est pas dans [1,2]",n)

    def ajouterPaire(self,forme1,forme2):
        self.classes1.ajouterPaire(forme1,forme2)
        self.classes2.ajouterPaire(forme2,forme1)
        
    def calculerClasses(self):
        return(self.classes1,self.classes2)

#### classesPaire

In [4]:
class classesPaire:
    '''
    Gestion des patrons, des classes et des transformations
    
    ajouterPatron : ajoute un patron et son motif associé (MGL)
    ajouterPaire : ajoute une paire de formes, calcule la classe de la forme1 et la règle sélectionnée
    sortirForme : cacule les formes de sortie correspondant à la forme1 avec leurs coefficients respectifs
    '''
    def __init__(self,case1,case2):
        self.case1=case1
        self.case2=case2
        self.nom=case1+"-"+case2
        self.classe={}
        self.nbClasse={}
        self.patrons={}
        self.entree={}
        self.sortie={}
        self.classeCF={}
        self.nbClasseCF={}
    
    def ajouterPatron(self,patron,motif):
        self.patrons[patron]=motif
        (entree,sortie)=patron.split("-")
        self.entree[patron]=entree.replace(u".",u"(.)")
        self.sortie[patron]=remplacementSortie(sortie)
    
    def ajouterPaire(self,forme1,forme2):
        '''
        on calcule la classe de la paire idClasseForme et la règle sélectionnée
        on incrémente le compteur de la classe et celui de la règle sélectionnée à l'intérieur de la classe
        '''
        classeFormeCF=[]
        regleFormeCF=""
        classeForme=[]
        regleForme=""
        for patron in self.patrons:
            filterF1=".*"+patron.split("-")[0]+"$"
            if re.match(filterF1,forme1):
                classeFormeCF.append(patron)
                if forme2==re.sub(self.entree[patron]+"$",self.sortie[patron],forme1):
                    regleFormeCF=patron
            filterF1=self.patrons[patron]
            if re.match(filterF1,forme1):
                classeForme.append(patron)
                '''
                le +"$" permet de forcer l'alignement à droite pour les transformations suffixales
                '''
                if forme2==re.sub(self.entree[patron]+"$",self.sortie[patron],forme1):
                    regleForme=patron
        idClasseFormeCF=", ".join(classeFormeCF)
        if not idClasseFormeCF in self.classeCF:
            self.classeCF[idClasseFormeCF]={}
            self.nbClasseCF[idClasseFormeCF]=0
        if not regleFormeCF in self.classeCF[idClasseFormeCF]:
            self.classeCF[idClasseFormeCF][regleFormeCF]=0
        self.nbClasseCF[idClasseFormeCF]+=1
        self.classeCF[idClasseFormeCF][regleFormeCF]+=1
        
        idClasseForme=", ".join(classeForme)
        if not idClasseForme in self.classe:
            self.classe[idClasseForme]={}
            self.nbClasse[idClasseForme]=0
        if not regleForme in self.classe[idClasseForme]:
            self.classe[idClasseForme][regleForme]=0
        self.nbClasse[idClasseForme]+=1
        self.classe[idClasseForme][regleForme]+=1

    def sortirForme(self,forme,contextFree=True):
        classeForme=[]
        sortieForme={}
        for patron in self.patrons:
            if contextFree:
                filterF1=".*"+patron.split("-")[0]+"$"
            else:
                filterF1=self.patrons[patron]
            # print filterF1,forme
            if re.match(filterF1,forme):
                classeForme.append(patron)
        if classeForme:
            idClasseForme=", ".join(classeForme)
            if contextFree:
                nbClasse=self.nbClasseCF
                classe=self.classeCF
            else:
                nbClasse=self.nbClasse
                classe=self.classe
            if idClasseForme in nbClasse:
                nTotal=nbClasse[idClasseForme]
                for patron in classe[idClasseForme]:
                    sortie=re.sub(self.entree[patron]+"$",self.sortie[patron],forme)
                    sortieForme[sortie]=float(classe[idClasseForme][patron])/nTotal
            else:
                if debug:
                    print (forme)
                    print ("pas de classe",idClasseForme)
                    print ("%.2f par forme de sortie" % (float(1)/len(classeForme)))
                nTotal=len(classeForme)
                for patron in classeForme:
                    sortie=re.sub(self.entree[patron]+"$",self.sortie[patron],forme)
                    sortieForme[sortie]=float(1)/nTotal
        else:
            if debug:
                print (forme) 
                print ("pas de patron")
        return sortieForme
        

In [5]:
def stemSpaceForm(rules,row,contextFree=False):
    swimSortir=rules.sortirForme(row,contextFree)
    if swimSortir:
        # print(swimSortir.keys(),list(swimSortir.keys()))
        result=list(swimSortir.keys())[0]
    else:
        result=np.nan
    return result

## Traitement

In [6]:
nums=range(5)
for num in nums:
    # rulesFile="vlexique2-S%d-omp-Regles.pkl"%num
    # resultFile="vlexique2-S%d-omp-Swim2.csv"%num
    # ompFile="vlexique2-S%d-omp.yaml"%num
    rulesFile="vlexique2-CV5-Train%d-omp-Regles.pkl"%num
    resultFile="vlexique2-CV5-Train%d-omp-Swim2.csv"%num
    ompFile="vlexique2-CV5-Train%d-omp.yaml"%num
    fRules=repFiles+rulesFile
    percentThreshold=.1
    debug=False
    
    if "omp" in rulesFile:
        bMorphomes=True
        with open(repFiles+ompFile) as inFile:
            dictMorphomeCases=yaml.safe_load(inFile)
    else:
        bMorphomes=False


    # print(dictMorphomeCases)

    dfSwim=pd.read_csv(repFiles+resultFile,sep=";",encoding="utf8")
    dfStemSpaces=dfSwim.copy()
    nbLexemes=dfSwim.lexeme.count()
    threshold=percentThreshold*nbLexemes
    print("seuil",threshold)

    with open(fRules, 'rb') as input:
        regles = pickle.load(input)

    zeroEntropy=[]
    for paire in regles:
        if paire[0]!=paire[1]:
            pClasses=regles[paire].classe
            bZero=True
            nbExemples=0
            for regle in pClasses:
                if len(pClasses[regle])>1:
                    bZero=False
                    break
                else:
                    for classe in pClasses[regle]:
                        nbExemples+=pClasses[regle][classe]
            if bZero and nbExemples>threshold:
                zeroEntropy.append((paire,nbExemples))

    # print(sorted(zeroEntropy, key=lambda tup: tup[1]))

    nbStemSpaceForms=0
    for (c1,c2),_ in zeroEntropy:
        # print(c1,c2)
        nbStemSpaceForms+=dfStemSpaces.loc[(dfStemSpaces[c1].notnull()) & (dfStemSpaces[c2].isnull()),c1].count()
        dfStemSpaces.loc[(dfStemSpaces[c1].notnull()) & (dfStemSpaces[c2].isnull()),c2]=dfStemSpaces.loc[(dfStemSpaces[c1].notnull()) & (dfStemSpaces[c2].isnull()),c1].apply(lambda x: stemSpaceForm(regles[c1,c2],x,contextFree=True))

    if bMorphomes:
        for c1 in dictMorphomeCases:
            for c2 in dictMorphomeCases[c1]:
                if c1!=c2:
                    # print(c1,"=>",c2)
                    dfStemSpaces[c2]=dfStemSpaces[c1]

    dfStemSpaces.to_csv(repFiles+resultFile.replace(".csv","-StemSpace.csv"),index=None,sep=";",encoding="utf8")
    print(resultFile,nbStemSpaceForms)
    ding()

seuil 499.1
vlexique2-CV5-Train0-omp-Swim2.csv 801
seuil 499.20000000000005
vlexique2-CV5-Train1-omp-Swim2.csv 1459
seuil 498.6
vlexique2-CV5-Train2-omp-Swim2.csv 1434
seuil 499.0
vlexique2-CV5-Train3-omp-Swim2.csv 560
seuil 498.90000000000003
vlexique2-CV5-Train4-omp-Swim2.csv 1417
